This notebook will go over differnt training models to answer the question "can we predict a binary claissfication on will a song be a "hit"/ successfull on soptify"

In [1]:
from pathlib import Path
# all data files paths
DATA_PATH =             Path("data/data.csv")
DATA_BY_ARTIST_PATH =   Path("data/data_by_artist.csv")
DATA_BY_GENRES_PATH =   Path("data/data_by_genres.csv")
DATA_BY_YEAR_PATH =     Path("data/data_by_year.csv")
DATA_W_GENRES_PATH =    Path("data/data_w_genres.csv")

In [2]:
import pandas as pd
# load file to dataframe
df = pd.read_csv(DATA_PATH)
df.head()

,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667,0.211,0,4BJqT0PrAfrxzMOxytFOIz,0.878000,10,0.665,-20.096,1,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4,1921,0.0366,80.954
1,0.9630,1921,0.732,['Dennis Day'],0.819,180533,0.341,0,7xPhfUan2yNtyFG0cUWkt8,0.000000,7,0.160,-12.441,1,Clancy Lowered the Boom,5,1921,0.4150,60.936
2,0.0394,1921,0.961,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062,0.166,0,1o6I8BglA6ylDMrIELygv1,0.913000,3,0.101,-14.850,1,Gati Bali,5,1921,0.0339,110.339
3,0.1650,1921,0.967,['Frank Parker'],0.275,210000,0.309,0,3ftBPsC5vPBKxYSee08FDH,0.000028,5,0.381,-9.316,1,Danny Boy,3,1921,0.0354,100.109
4,0.2530,1921,0.957,['Phil Regan'],0.418,166693,0.193,0,4d6HGyGT8e121BsdKmw9v6,0.000002,3,0.229,-10.096,1,When Irish Eyes Are Smiling,2,1921,0.0380,101.665


In [3]:
# dropping missing rows and dropping identifier columns
# dropping rows with missing values and dropping identifier columns
df = df.dropna()
cols_to_drop = ["id", "artists", "name"]
df = df.drop(columns=cols_to_drop, errors="ignore")
df.head()

,valence,year,acousticness,danceability,duration_ms,energy,explicit,instrumentalness,key,liveness,loudness,mode,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.982,0.279,831667,0.211,0,0.878000,10,0.665,-20.096,1,4,1921,0.0366,80.954
1,0.9630,1921,0.732,0.819,180533,0.341,0,0.000000,7,0.160,-12.441,1,5,1921,0.4150,60.936
2,0.0394,1921,0.961,0.328,500062,0.166,0,0.913000,3,0.101,-14.850,1,5,1921,0.0339,110.339
3,0.1650,1921,0.967,0.275,210000,0.309,0,0.000028,5,0.381,-9.316,1,3,1921,0.0354,100.109
4,0.2530,1921,0.957,0.418,166693,0.193,0,0.000002,3,0.229,-10.096,1,2,1921,0.0380,101.665


In [4]:
# setup "hit" feature
# "hit" feature is a binary feature to replace "popularity". if popularity is above a threshold, hit = 1 else hit = 0
POPULARITY_THRESHOLD = 50
# creating binary target column 'hit'
df["hit"] = (df["popularity"] >= POPULARITY_THRESHOLD).astype(int)
# getting hit value_counts
df["hit"].value_counts(normalize=True)

hit
0    0.772808
1    0.227192
Name: proportion, dtype: float64

In [5]:
# dropping redundent features and popularity feature
cols_to_drop = ["popularity", "release_date"]
df = df.drop(columns=cols_to_drop, errors="ignore")
df.head()

,valence,year,acousticness,danceability,duration_ms,energy,explicit,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,hit
0,0.0594,1921,0.982,0.279,831667,0.211,0,0.878000,10,0.665,-20.096,1,0.0366,80.954,0
1,0.9630,1921,0.732,0.819,180533,0.341,0,0.000000,7,0.160,-12.441,1,0.4150,60.936,0
2,0.0394,1921,0.961,0.328,500062,0.166,0,0.913000,3,0.101,-14.850,1,0.0339,110.339,0
3,0.1650,1921,0.967,0.275,210000,0.309,0,0.000028,5,0.381,-9.316,1,0.0354,100.109,0
4,0.2530,1921,0.957,0.418,166693,0.193,0,0.000002,3,0.229,-10.096,1,0.0380,101.665,0


In [6]:
# spliting data
from sklearn.model_selection import train_test_split
X = df.drop(columns=["hit"], errors="ignore")
y = df["hit"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# applying scalling to dataset
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
# evaluation metrics function
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

def evaluate_model(y_test, y_pred, print_report = False):
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    cr = classification_report(y_test, y_pred)
    
    if print_report:
        print("Accuracy:", round(acc, 4))
        print("Precision (weighted):", round(prec, 4))
        print("Recall (weighted):", round(rec, 4))
        print("F1 Score (weighted):", round(f1, 4))
        print("ROC AUC:", round(auc, 4))
        print("\nConfusion Matrix:")
        print(cm)
        print("\nClassification Report:")
        print(cr)
    
    return acc, prec, rec, f1, auc, cm, cr

In [9]:
# TODO later
def create_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    return 0

def create_classification_report(y_test, y_pred):
    cr = classification_report(y_test, y_pred)
    return 0

def create_confusion_matrix_and_classification_report(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    cr = classification_report(y_test, y_pred)
    return 0

In [10]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# training pipeline function
def training_pipeline(X_train, y_train, X_test, y_test):
    results = []
    
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42, n_jobs=-1),
        "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
        "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
        "SGD Classifier": SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42),
        "Extra Trees": ExtraTreesClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "LinearSVC": LinearSVC(class_weight="balanced", random_state=42),
        # "SVM (RBF)": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=11, n_jobs=-1),
        "Gaussian NB": GaussianNB(),
    }

    for name, model in models.items():
        print("training:", name)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        # y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

        acc, prec, rec, f1, auc, _, _ = evaluate_model(y_test, y_pred, print_report=False)

        results.append({
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "ROC AUC": auc,
            "y_test": y_test, # used for confusion_matrix later time
            "y_pred": y_pred
        })

    return pd.DataFrame(results)



In [11]:
def add_model_to_results(model_name, y_test, y_pred, results):
    acc, prec, rec, f1, auc, _, _ = evaluate_model(y_test, y_pred, print_report=False)
    results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "ROC AUC": auc,
        "y_test": y_test, # used for confusion_matrix later time
        "y_pred": y_pred
    })
    return results

In [12]:
def train_single_model(model, model_name, X_train, y_train, X_test, y_test, results_df):
    
    print("training:", model_name)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results_list = results_df.to_dict("records")
    results_list = add_model_to_results(model_name, y_test, y_pred, results_list)

    return pd.DataFrame(results_list)

In [13]:
base_data_results = training_pipeline(X_train, y_train, X_test, y_test)


training: Logistic Regression
training: Random Forest
training: Decision Tree
training: SGD Classifier
training: Extra Trees
training: Gradient Boosting
training: AdaBoost


c:\Users\longt\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


training: LinearSVC


c:\Users\longt\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\longt\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\longt\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\longt\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

training: KNN
training: Gaussian NB


In [14]:
base_data_results

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,y_test,y_pred
0,Logistic Regression,0.730450,0.803002,0.730450,0.749576,0.739238,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, ..."
1,Random Forest,0.863877,0.857753,0.863877,0.854979,0.755643,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Decision Tree,0.809323,0.809787,0.809323,0.809553,0.731392,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, ..."
3,SGD Classifier,0.758929,0.611398,0.758929,0.667219,0.494099,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Extra Trees,0.863819,0.857486,0.863819,0.855589,0.758748,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,Gradient Boosting,0.865108,0.858866,0.865108,0.857486,0.763177,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
6,AdaBoost,0.858867,0.852098,0.858867,0.852803,0.763800,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, ..."
7,LinearSVC,0.770795,0.594125,0.770795,0.671026,0.500000,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
8,KNN,0.807125,0.787814,0.807125,0.784189,0.645493,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9,Gaussian NB,0.822742,0.844810,0.822742,0.829950,0.799914,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, ..."


In [15]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd

# base estimators (all support predict_proba -> required for soft voting)
estimators = [
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ("rf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)),
    ("gb", GradientBoostingClassifier(random_state=42)),
]

voting_models = {
    "Voting (Hard)": VotingClassifier(estimators=estimators, voting="hard"),
    "Voting (Soft)": VotingClassifier(estimators=estimators, voting="soft")  # can also add weights=[1,2,2]
}

rows = []
for name, model in voting_models.items():
    print("training:", name)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc, prec, rec, f1, auc, cm, cr = evaluate_model(y_test, y_pred, print_report=False)
    rows.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "ROC AUC": auc,
        "y_test": y_test,
        "y_pred": y_pred
    })

voting_results = pd.DataFrame(rows).sort_values("F1 Score", ascending=False)
voting_results


training: Voting (Hard)


c:\Users\longt\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


training: Voting (Soft)


c:\Users\longt\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,y_test,y_pred
0,Voting (Hard),0.864932,0.858677,0.864932,0.858644,0.769800,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
1,Voting (Soft),0.858516,0.853911,0.858516,0.855482,0.780143,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."


In [ ]:
# training without year features 
X_train_clone = X_train.copy(deep=True)
X_test_clone = X_test.copy(deep=True)
X_train_no_year = X_train_clone.drop(columns=["year"], errors="ignore")
X_test_no_year = X_test_clone.drop(columns=["year"], errors="ignore")
no_year_data_results = training_pipeline(X_train_no_year, y_train, X_test_no_year, y_test)


training: Logistic Regression
training: Random Forest
training: Decision Tree
training: SGD Classifier
training: Extra Trees
training: Gradient Boosting
training: AdaBoost


c:\Users\longt\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


training: LinearSVC


c:\Users\longt\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\longt\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\longt\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\longt\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

training: KNN
training: Gaussian NB


In [15]:
no_year_data_results

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,y_test,y_pred
0,Logistic Regression,0.728780,0.802017,0.728780,0.748080,0.737661,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, ..."
1,Random Forest,0.821511,0.806937,0.821511,0.804411,0.677773,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
2,Decision Tree,0.753450,0.753395,0.753450,0.753422,0.651003,116368 0 161935 0 135703 0 112288 ...,"[1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, ..."
3,SGD Classifier,0.770648,0.594099,0.770648,0.670954,0.499905,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Extra Trees,0.820574,0.805740,0.820574,0.803354,0.676357,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
5,Gradient Boosting,0.819109,0.803921,0.819109,0.802341,0.676080,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
6,AdaBoost,0.810641,0.793116,0.810641,0.792694,0.662953,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
7,LinearSVC,0.770795,0.594125,0.770795,0.671026,0.500000,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
8,KNN,0.752688,0.682547,0.752688,0.694523,0.522834,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9,Gaussian NB,0.774955,0.736665,0.774955,0.695544,0.522818,116368 0 161935 0 135703 0 112288 ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
# generation split (igonore for now)
baby_boomers = [1946, 1964]
gen_x = [1965, 1980]
millennials = [1981, 1996]
gen_z = [1997, 2012]
gen_a = [2013, 2024]
gen_b = [2025, 2039] 

In [17]:
# hyperparamerter tunning
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform
import pandas as pd

def hyperparameter_tuning(X_train, y_train, X_test, y_test, n_iter=20, cv=5):
    cv_strategy = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    searches = {
        "Logistic Regression": (
            Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(class_weight="balanced", random_state=42, max_iter=3000))
            ]),
            {
                "clf__C": loguniform(1e-3, 1e2),
                "clf__solver": ["lbfgs", "liblinear"]
            }
        ),
        "Random Forest": (
            RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
            {
                "n_estimators": [200, 300, 500, 800],
                "max_depth": [None, 10, 20, 30, 50],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4],
                "max_features": ["sqrt", "log2", None]
            }
        ),
        "Decision Tree": (
            DecisionTreeClassifier(class_weight="balanced", random_state=42),
            {
                "max_depth": [None, 5, 10, 20, 30],
                "min_samples_split": [2, 5, 10, 20],
                "min_samples_leaf": [1, 2, 4, 8],
                "criterion": ["gini", "entropy", "log_loss"]
            }
        ),
        "SGD Classifier": (
            Pipeline([
                ("scaler", StandardScaler()),
                ("clf", SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42))
            ]),
            {
                "clf__alpha": loguniform(1e-6, 1e-2),
                "clf__penalty": ["l2", "l1", "elasticnet"],
                "clf__max_iter": [1000, 2000, 3000]
            }
        ),
    }

    tuned_results = []

    for name, (estimator, param_dist) in searches.items():
        print(f"Tuning: {name}")

        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_dist,
            n_iter=n_iter,
            scoring="f1_weighted",
            cv=cv_strategy,
            n_jobs=-1,
            random_state=42,
            verbose=1
        )

        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        y_pred = best_model.predict(X_test)

        acc, prec, rec, f1, auc, _, _ = evaluate_model(y_test, y_pred, print_report=False)

        tuned_results.append({
            "Model": name,
            "Best Params": search.best_params_,
            "CV Best F1 (weighted)": search.best_score_,
            "Test Accuracy": acc,
            "Test Precision": prec,
            "Test Recall": rec,
            "Test F1": f1,
            "Test ROC AUC": auc
        })

    return pd.DataFrame(tuned_results).sort_values("Test F1", ascending=False)


In [ ]:
tuned_results = hyperparameter_tuning(X_train, y_train, X_test, y_test, n_iter=10, cv=5)
tuned_results


Tuning: Logistic Regression
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Tuning: Random Forest
Fitting 5 folds for each of 10 candidates, totalling 50 fits
